# DurianAI — KnockNet-lite Acoustic Model Training

> Phase 1: CNN on Mel Spectrogram → TFLite INT8 for browser deployment

**Data sources:**
- Zenodo Multi-Modal (189 samples, CC BY) — 3-class: Unripe/Ripe/Overripe
- Dalvii GitHub (100 WAV, Dona variety) — 2-class: 75-85%/95%-Ripe

**Target:** 80%+ accuracy, ~1-5MB TFLite model

In [ ]:
#@title 1. Setup Environment { display-mode: "form" }
!pip install -q librosa soundfile scikit-learn tensorflow

import os
import numpy as np
import librosa
import soundfile as sf
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from pathlib import Path
from collections import Counter

print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
#@title 2. Download Data { display-mode: "form" }

# === Zenodo Multi-Modal Dataset ===
import urllib.request, json, zipfile

ZENODO_DIR = '/content/durian_data/zenodo'
DALVII_DIR = '/content/durian_data/dalvii'
os.makedirs(ZENODO_DIR, exist_ok=True)
os.makedirs(DALVII_DIR, exist_ok=True)

# Download Zenodo CSV (tiny, for labels)
csv_url = 'https://zenodo.org/records/18603796/files/durian_characteristics_cleaned.csv?download=1'
csv_path = os.path.join(ZENODO_DIR, 'durian_characteristics_cleaned.csv')
if not os.path.exists(csv_path):
    urllib.request.urlretrieve(csv_url, csv_path)
print(f'CSV downloaded: {os.path.getsize(csv_path)/1024:.1f} KB')

# Download Zenodo audio (1.5 GB — takes ~5 min)
audio_url = 'https://zenodo.org/records/18603796/files/dataset_clean_sound.zip?download=1'
audio_zip = os.path.join(ZENODO_DIR, 'dataset_clean_sound.zip')
if not os.path.exists(audio_zip):
    print('Downloading Zenodo audio (1.5 GB)...')
    urllib.request.urlretrieve(audio_url, audio_zip)
    print('Done!')

# Extract audio
audio_extract = os.path.join(ZENODO_DIR, 'sound')
if not os.path.exists(audio_extract):
    with zipfile.ZipFile(audio_zip, 'r') as z:
        z.extractall(audio_extract)
    print(f'Extracted to {audio_extract}')

# === Dalvii GitHub Dataset ===
import json
GITHUB_API = 'https://api.github.com/repos/Dalvii/durian-maturity-classification/contents/AUDIO_DATA'
req = urllib.request.Request(GITHUB_API)
req.add_header('Accept', 'application/vnd.github.v3+json')
try:
    with urllib.request.urlopen(req, timeout=30) as resp:
        files = json.loads(resp.read().decode())
    wav_files = [f for f in files if f['name'].endswith('.wav')]
    print(f'Found {len(wav_files)} Dalvii WAV files')
    for f in wav_files:
        filepath = os.path.join(DALVII_DIR, f['name'])
        if not os.path.exists(filepath):
            urllib.request.urlretrieve(f['download_url'], filepath)
    print('Dalvii data downloaded!')
except Exception as e:
    print(f'GitHub API failed: {e}')
    print('Fallback: git clone')
    os.system('git clone https://github.com/Dalvii/durian-maturity-classification.git /content/durian_data/dalvii_repo')
    # Copy AUDIO_DATA

In [ ]:
#@title 3. Parse Labels & Explore { display-mode: "form" }
import csv

# Load Zenodo CSV
zenodo_labels = {}  # code → ripeness
with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        code = row['Code']
        ripeness = row['Ripeness']  # Unripe/Ripe/Overripe
        # Use Actual_Ripening_Status as ground truth (skip Disease/empty)
        actual = row['Actual_Ripening_Status']
        if actual in ('Unripe', 'Ripe', 'Overripe'):
            zenodo_labels[code] = actual.lower()  # unripe/ripe/overripe
        elif ripeness in ('Unripe', 'Ripe', 'Overripe'):
            zenodo_labels[code] = ripeness.lower()

print(f'Zenodo labeled samples: {len(zenodo_labels)}')
label_dist = Counter(zenodo_labels.values())
print(f'Distribution: {dict(label_dist)}')

# Parse Dalvii labels from filename
dalvii_labels = {}  # filename → label
for wav_path in Path(DALVII_DIR).glob('*.wav'):
    name = wav_path.stem
    if '75-85' in name or '75' in name:
        dalvii_labels[name] = 'unripe'
    elif '95' in name or 'Ripe' in name or 'ripe' in name:
        dalvii_labels[name] = 'ripe'

print(f'Dalvii labeled samples: {len(dalvii_labels)}')
d_dist = Counter(dalvii_labels.values())
print(f'Distribution: {dict(d_dist)}')

In [ ]:
#@title 4. Extract Features & Build Dataset { display-mode: "form" }

TARGET_SR = 16000
N_MELS = 40
N_FFT = 512
HOP_LENGTH = 256
SEGMENT_DURATION = 2.0  # seconds

def extract_mel(y, sr=TARGET_SR):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)

def load_and_segment(path, duration=SEGMENT_DURATION):
    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)
    if np.max(np.abs(y)) > 0:
        y = y / np.max(np.abs(y))
    seg_len = int(duration * TARGET_SR)
    segments = []
    if len(y) < seg_len:
        segments.append(np.pad(y, (0, seg_len - len(y)), mode='constant'))
    else:
        # Use overlapping windows with 50% overlap for more data
        step = seg_len // 2
        for i in range(0, len(y) - seg_len + 1, step):
            segments.append(y[i:i + seg_len])
    return segments

all_X = []
all_y = []

# Process Zenodo audio
print('Processing Zenodo audio...')
audio_files = list(Path(audio_extract).rglob('*.wav'))
print(f'Found {len(audio_files)} Zenodo WAV files')

# Zenodo file naming convention matches the Code field
# e.g. IM_CA_UN_1.wav or similar
zenodo_matched = 0
for wav_path in audio_files:
    name = wav_path.stem
    # Try to match with zenodo_labels
    # The Code field format: IM_CA_UN_1, M_CB_RI_92, OM_CC_OR_189
    # Try direct match, then partial match
    matched_label = None
    for code, label in zenodo_labels.items():
        if code in name or name in code:
            matched_label = label
            break
    # Fallback: parse from filename
    if matched_label is None:
        if '_UN_' in name.upper() or 'UN' in name.upper():
            matched_label = 'unripe'
        elif '_RI_' in name.upper() or 'RI' in name.upper():
            matched_label = 'ripe'
        elif '_OR_' in name.upper() or 'OR' in name.upper():
            matched_label = 'overripe'
    
    if matched_label is None:
        continue
    
    try:
        segments = load_and_segment(str(wav_path))
        for seg in segments:
            feat = extract_mel(seg)
            all_X.append(feat)
            all_y.append(matched_label)
        zenodo_matched += 1
    except Exception as e:
        print(f'  [FAIL] {name}: {e}')

print(f'Zenodo: {zenodo_matched} files → {len([l for l in all_y])} segments')

# Process Dalvii audio
print('Processing Dalvii audio...')
dalvii_matched = 0
for wav_path in Path(DALVII_DIR).glob('*.wav'):
    name = wav_path.stem
    label = dalvii_labels.get(name)
    if label is None:
        continue
    try:
        segments = load_and_segment(str(wav_path))
        for seg in segments:
            feat = extract_mel(seg)
            all_X.append(feat)
            all_y.append(label)
        dalvii_matched += 1
    except Exception as e:
        print(f'  [FAIL] {name}: {e}')

print(f'Dalvii: {dalvii_matched} files → {len(all_y)} total segments')

X = np.array(all_X)[..., np.newaxis]  # Add channel dim
y = np.array(all_y)

dist = Counter(y)
print(f'\nFinal dataset: {X.shape}')
print(f'Classes: {dict(dist)}')

In [ ]:
#@title 5. Audio Data Augmentation { display-mode: "form" }
augment_factor = 3 #@param {type:"slider", min:1, max:10, step:1}

def augment_mel(mel_spec):
    """SpecAugment: time masking + frequency masking."""
    augmented = mel_spec.copy()
    # Frequency masking (mask up to 8 consecutive mel bins)
    f_mask_width = np.random.randint(1, 8)
    f_start = np.random.randint(0, augmented.shape[0] - f_mask_width)
    augmented[f_start:f_start+f_mask_width, :] = 0
    # Time masking (mask up to 16 consecutive time frames)
    t_mask_width = np.random.randint(1, 16)
    t_start = np.random.randint(0, augmented.shape[1] - t_mask_width)
    augmented[:, t_start:t_start+t_mask_width] = 0
    return augmented

if augment_factor > 1:
    aug_X = []
    aug_y = []
    for i in range(len(X)):
        aug_X.append(X[i])
        aug_y.append(y[i])
        for _ in range(augment_factor - 1):
            aug_X.append(augment_mel(X[i])[..., np.newaxis])
            aug_y.append(y[i])
    X = np.array(aug_X)
    y = np.array(aug_y)
    print(f'Augmented dataset: {X.shape}')
    print(f'Classes: {dict(Counter(y))}')

In [ ]:
#@title 6. Split & Encode Labels { display-mode: "form" }

# Unified label encoding
LABEL_NAMES = ['unripe', 'ripe', 'overripe']
label_map = {name: i for i, name in enumerate(LABEL_NAMES)}

y_encoded = np.array([label_map[l] for l in y])

# Stratified split: 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f'Train: {X_train.shape} — {dict(Counter(y_train))}')
print(f'Val:   {X_val.shape} — {dict(Counter(y_val))}')
print(f'Test:  {X_test.shape} — {dict(Counter(y_test))}')
print(f'\nLabels: {LABEL_NAMES}')

In [ ]:
#@title 7. Build CNN Model { display-mode: "form" }

def build_knocknet_lite(input_shape, num_classes=3):
    model = keras.Sequential([
        layers.Input(shape=input_shape),
        # Block 1
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        # Block 2
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        # Block 3
        layers.Conv2D(128, (3,3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        # Head
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])
    return model

model = build_knocknet_lite(X_train.shape[1:], num_classes=len(LABEL_NAMES))
model.summary()

In [ ]:
#@title 8. Train { display-mode: "form" }
epochs = 100 #@param {type:"integer"}
batch_size = 32 #@param {type:"integer"}
learning_rate = 0.001 #@param {type:"number"}

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=epochs,
    batch_size=batch_size,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(
            '/content/durian_acoustic_best.keras',
            save_best_only=True, monitor='val_accuracy'
        ),
    ],
)

In [ ]:
#@title 9. Evaluate { display-mode: "form" }
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_acc:.4f}')

# Confusion matrix
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_classes)
print('\nConfusion Matrix:')
print(cm)

print('\nClassification Report:')
print(classification_report(y_test, y_pred_classes, target_names=LABEL_NAMES))

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()
plt.tight_layout()
plt.savefig('/content/acoustic_training_history.png')
plt.show()

In [ ]:
#@title 10. Export TFLite { display-mode: "form" }

# INT8 quantization (smallest, fastest)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
tflite_int8 = converter.convert()

int8_path = '/content/knocknet_lite.tflite'
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)
print(f'TFLite INT8: {len(tflite_int8)/1024:.1f} KB')

# FP16 quantization (better accuracy)
converter_fp16 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_fp16.optimizations = [tf.lite.Optimize.DEFAULT]
converter_fp16.target_spec.supported_types = [tf.float16]
tflite_fp16 = converter_fp16.convert()

fp16_path = '/content/knocknet_lite_fp16.tflite'
with open(fp16_path, 'wb') as f:
    f.write(tflite_fp16)
print(f'TFLite FP16: {len(tflite_fp16)/1024:.1f} KB')

# Save labels
with open('/content/acoustic_labels.txt', 'w') as f:
    for label in LABEL_NAMES:
        f.write(label + '\n')
print(f'Labels saved: {LABEL_NAMES}')

# Verify TFLite inference
interpreter = tf.lite.Interpreter(model_path=int8_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print(f'\nTFLite input shape: {input_details[0]["shape"]}')
print(f'TFLite output shape: {output_details[0]["shape"]}')

# Test on one sample
test_idx = 0
interpreter.set_tensor(input_details[0]['index'], X_test[test_idx:test_idx+1])
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
print(f'Sample prediction: {LABEL_NAMES[np.argmax(output)]} (true: {LABEL_NAMES[y_test[test_idx]]})')

In [ ]:
#@title 12. Export TF.js SavedModel (for browser-side acoustic inference) { display-mode: "form" }

# Install tensorflowjs converter
!pip install -q tensorflowjs

# Export acoustic model in TF.js format (optional — for browser-side inference)
# Primary deployment is TFLite on backend, but TF.js enables browser-side fallback

import subprocess, shutil, json

tfjs_dir = '/content/knocknet_lite_tfjs'
print('Converting acoustic model to TF.js format...')

# Save as SavedModel first
sm_dir = '/content/knocknet_lite_savedmodel'
model.save(sm_dir)

result = subprocess.run(
    ['tensorflowjs_converter',
     '--input_format=tf_saved_model',
     '--output_format=tfjs_graph_model',
     sm_dir,
     tfjs_dir],
    capture_output=True, text=True
)

if result.returncode != 0:
    print('ERROR:', result.stderr)
else:
    print('TF.js acoustic model saved ✓')

# List output files
for f in os.listdir(tfjs_dir):
    size = os.path.getsize(os.path.join(tfjs_dir, f))
    print(f'  {f}: {size/1024:.1f} KB')

# Save metadata
metadata = {
    'version': '1.0',
    'created_at': str(np.datetime64('today')),
    'model_type': 'KnockNet-lite-CNN',
    'classes': LABEL_NAMES,
    'input_shape': [1, N_MELS, None, 1],
    'notes': 'Phase 1 acoustic model — mel spectrogram CNN'
}
with open(os.path.join(tfjs_dir, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

# Copy labels
shutil.copy('/content/acoustic_labels.txt', os.path.join(tfjs_dir, 'labels.txt'))

print(f'\nTF.js acoustic model ready at: {tfjs_dir}')
print('Upload to frontend/public/models/acoustic/ for optional browser-side inference')
print('(Backend TFLite is the primary deployment path)')

In [ ]:
#@title 13. Download All Results { display-mode: "form" }
from google.colab import files
import os

# Download TFLite files (backend deployment)
files.download(int8_path)
files.download(fp16_path)
files.download('/content/acoustic_labels.txt')
files.download('/content/acoustic_training_history.png')

# Download TF.js model files (optional browser-side inference)
tfjs_dir = '/content/knocknet_lite_tfjs'
if os.path.isdir(tfjs_dir):
    for f in os.listdir(tfjs_dir):
        filepath = os.path.join(tfjs_dir, f)
        if os.path.getsize(filepath) < 100 * 1024 * 1024:
            print(f'Downloading: {f}')
            files.download(filepath)

print('\n=== Deployment Guide ===')
print('1. knocknet_lite.tflite → backend/models/acoustic/')
print('   (primary: backend TFLite inference via tflite-runtime)')
print('2. TF.js files → frontend/public/models/acoustic/')
print('   (optional: browser-side fallback inference)')
print('3. Git push → Render auto-deploy → POST /api/model/reload ✓')